In [ ]:
%reset -f

In [30]:
import os
import cv2
import imagehash
import pytesseract
import numpy as np
from glob import glob
import shutil
from PIL import Image
import re
from difflib import SequenceMatcher

In [45]:
def create_dir(path):
    if not os.path.exists(path):
        os.makedirs(path)
        return True
    return False
        
def remove_dir(path_dir):
    if os.path.exists(path_dir):
        shutil.rmtree(path_dir)
        return True
    else:
        return False

import os


def save_text_to_file(text,output_dir,file_name):

    os.makedirs(output_dir, exist_ok=True)
    file_path = os.path.join(output_dir,file_name)

    with open(file_path,"w",encoding="utf-8" ) as file:
        file.write(text)

    return file_path

In [35]:
def Video_To_frames(video_path, output_path):
    video_name      = os.path.splitext(os.path.basename(video_path))[0]
    save_path       = os.path.join(output_path, video_name)

    create_dir(save_path)
    cap = cv2.VideoCapture(video_path)

    prev_gray  = None
    prev_img   = None
    page_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        if prev_gray is None:
            cv2.imwrite(os.path.join(save_path, f"page_{page_count}.png"), frame)
            page_count += 1
            prev_gray = gray
            prev_img = frame
            continue

        diff = cv2.absdiff(prev_gray, gray)
        score = diff.mean()

        if score != 0:
            cv2.imwrite(os.path.join(save_path, f"page_{page_count}.png"), prev_img)
            prev_img = frame
            prev_gray = gray
            page_count += 1
    cap.release()
    return save_path

In [36]:
def filter_duplicate_images(frames_path,hash_threshold=5):

    unique_dir = os.path.join(frames_path, "unique_pages" )

    os.makedirs(unique_dir, exist_ok=True)

    prev_hash = None

    page_count = 0
    count = 0

    for image_file in  os.listdir(frames_path):

        if not image_file.endswith(".png"):
            continue
        
        image_path   = os.path.join(frames_path, f"page_{page_count}.png")
        img          = Image.open(image_path)
        current_hash = imagehash.phash(img)

        if prev_hash is None:

            save_path = os.path.join(unique_dir, f"page_{count}.png" )
            shutil.copy(image_path, save_path)
            prev_hash = current_hash
            page_count += 1
            count += 1
            continue

        hash_diff = current_hash - prev_hash
        
        if hash_diff > hash_threshold:

            save_path = os.path.join(unique_dir,f"page_{count}.png")
            shutil.copy(image_path, save_path)
            prev_hash = current_hash
            count += 1  
        page_count += 1

    return unique_dir

In [54]:
def clean_text(text):

    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def get_page_signature(text, length=150):

    text = clean_text(text)
    return text[:length]


def extract_unique_text(frames_path, similarity_threshold=0.60):

    previous_signature = ""

    myconfig = r'--oem 3 --psm 6'

    all_text = ""

    cout = 0

    for image_file in os.listdir(frames_path):

        if not image_file.endswith(".png"):
            continue
            
        image_path = os.path.join(frames_path, f"page_{cout}.png")
        img        = Image.open(image_path)
        text       = pytesseract.image_to_string(img, config=myconfig )
        signature  = get_page_signature(text)
        similarity = SequenceMatcher(None,signature, previous_signature).ratio()

        if similarity < similarity_threshold:

            all_text          += "\n\n" + text
            previous_signature = signature

        cout += 1

    return all_text

In [55]:
video_Folder_path     = r"C:\Users\MIL\Videos\OBS"
img_path       = r"C:\Users\MIL\Documents\OUTPUTS\OutPut Images"
txt_path       = r"C:\Users\MIL\Documents\OUTPUTS\OutPut Files"

matches = glob(video_Folder_path)
if matches:
    for folder in matches:
        for item in glob(os.path.join(folder, "*")):
            video_name      = os.path.splitext(os.path.basename(item))[0]
            print(item," ",video_name)
            frames_dir = Video_To_frames(item, img_path) 
            unique_dir = filter_duplicate_images(frames_dir)
            book_text  = extract_unique_text(unique_dir)
            #remove_dir(frames_dir)
            save_text_to_file(book_text,txt_path,video_name+".txt")

C:\Users\MIL\Videos\OBS\Chapter 1.mp4   Chapter 1
C:\Users\MIL\Videos\OBS\chapter 2.mp4   chapter 2
